# Project - Airline AI Assistant

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialization
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")



In [ ]:
# Constants
MODEL_OLLAMA = "gpt-oss:20b"
MODEL_GEMINI = "gemma3:270m"

In [ ]:
ollama = OpenAI(
    api_key="ollama",
    base_url="http://127.0.0.1:11434/v1"
)

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chat(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]
    messages = [
        {"role": "system", "content": system_message},
        *history,
        {"role": "user", "content": message}
    ]
    response = ollama.chat.completions.create(
        model=MODEL_OLLAMA,
        messages=messages
    )
    return response.choices[0].message.content


# gr.ChatInterface(fn=chat, type="messages").launch()

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [ ]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    ticket_price = ticket_prices.get(destination_city.lower(), "Unknwon Ticket price")
    return f"The price of the ticket to {destination_city} is {ticket_price}"

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

In [ ]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def chat2(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = [
        {"role": "system", "content": system_message},
        *history,
        {"role": "user", "content": message}
    ]

    response = ollama.chat.completions.create(
        model=MODEL_OLLAMA,
        messages=messages,
        tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message

        tool_response = handle_tool_call(assistant_message)

        messages.append(assistant_message)
        messages.append(tool_response)

        response = ollama.chat.completions.create(
            model=MODEL_OLLAMA,
            messages=messages
        )

    return response.choices[0].message.content

In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
        return response
    return {}

In [ ]:
# gr.ChatInterface(fn=chat2, type="messages").launch()

## Handle Multiple Tool Calls

In [ ]:
# This is useful when user wants to know price of the tickets to multiple cities in one go, like london, paris
def handle_tool_calls(message):
    print("multi_tool_chats", message)
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('city_price')
    return responses

In [ ]:
def chat3(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = [
        {"role": "system", "content": system_message},
        *history,
        {"role": "user", "content": message}
    ]

    response = ollama.chat.completions.create(
        model=MODEL_OLLAMA,
        messages=messages,
        tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message

        tool_response = handle_tool_calls(assistant_message)

        messages.append(assistant_message)
        messages.extend(tool_response)

        response = ollama.chat.completions.create(
            model=MODEL_OLLAMA,
            messages=messages
        )

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat3, type="messages").launch()

In [ ]:
# Chat3 will fail on if customer asks to perform multiple tools calls based on condition, like first fetch price to a city and check if it's under 1000 then fetch another
# so that's why multi tool calls handling is needed
def chat4(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = [
        {"role": "system", "content": system_message},
        *history,
        {"role": "user", "content": message}
    ]
    response = ollama.chat.completions.create(
        model=MODEL_OLLAMA,
        messages=messages,
        tools=tools
    )
    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message

        tool_response = handle_tool_calls(assistant_message)

        messages.append(assistant_message)
        messages.extend(tool_response)
        
        response = ollama.chat.completions.create(
            model=MODEL_OLLAMA,
            messages=messages,
            tools=tools
        )
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat4, type="messages").launch()

## Using SQLite3 as database to get and set price

In [ ]:
import sqlite3

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("London")

In [ ]:
# gr.ChatInterface(fn=chat4, type="messages", title="FlightAI").launch()

In [ ]:
# Adding another tool set ticket price
def handle_tool_calls_brute(message):
    print("multi_tool_chats", message)
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

        elif tool_call.function.name == "set_ticket_price":  # brute force approach
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            set_ticket_price(city, float(price))
            responses.append({
                "role": "tool",
                "content": f"Fare price set {price} for city {city}",
                "tool_call_id": tool_call.id
            })
            
    return responses

In [ ]:
set_ticket_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The fare price of the  that the customer wants to travel to",
            },
        },
        "required": ["destination_city", 'price'],
        "additionalProperties": False,
    },
}

In [ ]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}, {"type": "function", "function": set_ticket_price_function}]

In [ ]:
gr.ChatInterface(fn=chat4, type="messages", title="FlightAI").launch()

In [ ]:
# Handle tool calls in pythonic way


def handle_get_ticket_price(destination_city): # this is unnecessary if we change the schema to accept city but kept it for meanwhile
    return get_ticket_price(city=destination_city)


def handle_set_ticket_price(destination_city, price): # this is unnecessary if we change the schema to accept city but kept it for meanwhile
    return set_ticket_price(
        city=destination_city,
        price=price
    )


tool_functions = {
    "get_ticket_price": handle_get_ticket_price,
    "set_ticket_price": handle_set_ticket_price,
}

def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        try:
            function = tool_functions[function_name]
        except KeyError:
            raise ValueError(f"Unknown tool: {function_name}")

        result = function(**arguments)

        responses.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result),
        })

    return responses